# Demo: MCDA pipeline (GEESP-Angola)
This notebook demonstrates loading the generated maps, running a simple MCDA weighted overlay, and saving a result.

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path('data/processed')
maps = {}
for name in ['mapa_irradiacao','mapa_populacao','mapa_distanciarede','mapa_declividade','mapa_ndvi']:
    p = DATA_DIR / f'{name}.npy'
    if p.exists():
        maps[name] = np.load(p)

# simple weights example
weights = {'mapa_irradiacao':0.25,'mapa_populacao':0.25,'mapa_distanciarede':0.2,'mapa_declividade':0.15,'mapa_ndvi':0.15}

# normalize and overlay
def norm(a):
    a = np.array(a,dtype=float)
    valid = np.isfinite(a)
    if valid.any():
        amin = a[valid].min()
        amax = a[valid].max()
        return np.nan_to_num((a-amin)/(amax-amin+1e-9))
    return np.zeros_like(a)

overlay = None
for k, arr in maps.items():
    w = weights.get(k, 0)
    n = norm(arr)
    if overlay is None:
        overlay = w * n
    else:
        overlay = overlay + w * n

# save and show
out = DATA_DIR / 'notebook_mapa_aptidao.npy'
np.save(out, overlay)
plt.imshow(overlay, cmap='viridis')
plt.colorbar()
plt.title('MCDA Overlay (demo)')
plt.show()